In [46]:
%pip install xlrd

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import pandas as pd

baseline = pd.read_excel(
    "../data/sief_bayley_baseline_aug2015.xls"
)

print("Dataset shape:", baseline.shape)
print("\nColumn names:")
print(baseline.columns.tolist())

Dataset shape: (2574, 27)

Column names:
['V001', 'treatment', 'cc_code', 'upazila', 'V001a', 'V5', 'V16', 'V17', 'V18', 'V21y', 'V21m', 'V22m', 'V23', 'RC_a', 'RC_b', 'EC_a', 'EC_b', 'Sum_a', 'Sum_b', 'TRS_C', 'TRS_RL', 'TRS_EL', 'AM001', 'AM002', 'AM003', 'AM004', 'AM005']


In [48]:
display(baseline.head())

,V001,treatment,cc_code,upazila,V001a,V5,V16,V17,V18,V21y,...,Sum_a,Sum_b,TRS_C,TRS_RL,TRS_EL,AM001,AM002,AM003,AM004,AM005
0,540,1,1,1,4,2,36,6,80,1.0,...,14,83,36,11,14,62.200001,53.099998,NaN,70.900002,44.200001
1,538,1,1,1,4,1,43,6,80,1.0,...,14,83,43,14,18,57.500000,47.500000,NaN,79.400002,46.299999
2,552,1,1,1,2,1,23,8,90,NaN,...,15,86,23,8,5,49.400002,42.900002,NaN,62.599998,42.200001
3,546,1,1,1,4,2,25,7,85,NaN,...,19,97,25,8,11,44.799999,37.299999,NaN,67.000000,44.299999
4,537,1,1,1,2,1,30,9,95,NaN,...,24,112,30,11,14,43.000000,35.299999,NaN,66.300003,41.400002


In [49]:
import pandas as pd

endline = pd.read_csv(
    "../data/BGD_2015_SCECSPIE-EL_v01_M_CSV/SIEF_endline_bayley.csv"
)

print("Endline shape:", endline.shape)

Endline shape: (2486, 36)


In [50]:
print("Endline outcome summary:")

display(
    endline[["Cog_Total", "Rec_Total", "Exp_Total"]]
    .describe()
    .round(2)
)

Endline outcome summary:


,Cog_Total,Rec_Total,Exp_Total
count,2486.00,2486.00,2486.00
mean,64.83,30.18,32.01
std,5.31,5.26,4.69
min,0.00,0.00,0.00
25%,63.00,26.00,30.00
50%,65.00,30.00,32.00
75%,67.00,34.00,34.00
max,81.00,47.00,47.00


In [51]:
print("Baseline unique IDs:", baseline["V001"].nunique())
print("Endline unique IDs:", endline["V001"].nunique())

print(
    "Matching child IDs:",
    baseline["V001"].isin(endline["V001"]).sum()
)

print("\nBaseline treatment counts:")
print(baseline["treatment"].value_counts().sort_index())

print("\nEndline treatment counts:")
print(endline["treat"].value_counts().sort_index())

Baseline unique IDs: 2574
Endline unique IDs: 2486
Matching child IDs: 2486

Baseline treatment counts:
treatment
0    1287
1    1287
Name: count, dtype: int64

Endline treatment counts:
treat
0    1236
1    1250
Name: count, dtype: int64


In [52]:
# Match baseline and endline using child ID
matched = baseline.merge(
    endline[["V001", "treat"]],
    on="V001",
    how="inner",
    validate="one_to_one"
)

# Compare treatment assignment
matched["treatment_match"] = (
    matched["treatment"] == matched["treat"]
)

print("Matched rows:", len(matched))
print("\nTreatment assignment consistency:")
print(matched["treatment_match"].value_counts())

print("\nTreatment mismatch rows:")
display(
    matched.loc[
        ~matched["treatment_match"],
        ["V001", "treatment", "treat"]
    ].head(10)
)

Matched rows: 2486

Treatment assignment consistency:
treatment_match
True    2486
Name: count, dtype: int64

Treatment mismatch rows:


,V001,treatment,treat


In [53]:
baseline_summary = baseline.groupby("treatment")[
    ["TRS_C", "TRS_RL", "TRS_EL"]
].agg(["mean", "std", "count"]).round(2)

display(baseline_summary)

TRS_C             TRS_RL             TRS_EL            
            mean   std count   mean   std count   mean   std count
treatment                                                         
0          39.89  9.14  1287  13.91  4.45  1287  14.38  5.75  1287
1          39.29  8.99  1287  13.70  4.34  1287  14.14  5.51  1287

In [54]:
from scipy.stats import ttest_ind

outcomes = ["TRS_C", "TRS_RL", "TRS_EL"]

for col in outcomes:
    control = baseline.loc[baseline["treatment"] == 0, col]
    treatment = baseline.loc[baseline["treatment"] == 1, col]

    t_stat, p_value = ttest_ind(
        treatment,
        control,
        equal_var=False
    )

    difference = treatment.mean() - control.mean()

    print(f"\nOutcome: {col}")
    print(f"Control Mean: {control.mean():.3f}")
    print(f"Treatment Mean: {treatment.mean():.3f}")
    print(f"Difference (T - C): {difference:.3f}")
    print(f"P-value: {p_value:.4f}")


Outcome: TRS_C
Control Mean: 39.893
Treatment Mean: 39.287
Difference (T - C): -0.605
P-value: 0.0904

Outcome: TRS_RL
Control Mean: 13.908
Treatment Mean: 13.702
Difference (T - C): -0.205
P-value: 0.2365

Outcome: TRS_EL
Control Mean: 14.375
Treatment Mean: 14.141
Difference (T - C): -0.234
P-value: 0.2922


In [55]:
baseline_endline = baseline.merge(
    endline,
    on="V001",
    how="inner",
    suffixes=("_baseline", "_endline"),
    validate="one_to_one"
)

print("Merged dataset shape:", baseline_endline.shape)

print("\nTreatment counts:")
print(baseline_endline["treatment"].value_counts())

print("\nTreatment consistency:")
print(
    (
        baseline_endline["treatment"]
        == baseline_endline["treat"]
    ).value_counts()
)

Merged dataset shape: (2486, 62)

Treatment counts:
treatment
1    1250
0    1236
Name: count, dtype: int64

Treatment consistency:
True    2486
Name: count, dtype: int64


In [56]:
print(baseline_endline.columns.tolist())

['V001', 'treatment', 'cc_code_baseline', 'upazila_baseline', 'V001a', 'V5', 'V16', 'V17', 'V18', 'V21y', 'V21m', 'V22m', 'V23', 'RC_a', 'RC_b', 'EC_a', 'EC_b', 'Sum_a', 'Sum_b', 'TRS_C', 'TRS_RL', 'TRS_EL', 'AM001_baseline', 'AM002_baseline', 'AM003_baseline', 'AM004_baseline', 'AM005_baseline', 'cc_code_endline', 'upazila_endline', 'district', 'union', 'treat', 'tc', 'TC_other', 'V004', 'V005', 'V016', 'V017', 'V018', 'V021_Y', 'V021_M', 'V022_M', 'V023', 'RC_1', 'RC_2', 'EC_1', 'EC_2', 'Sum_1', 'Sum_2', 'V024', 'V025', 'V026', 'V027', 'V028', 'Cog_Total', 'Rec_Total', 'Exp_Total', 'AM001_endline', 'AM002_endline', 'AM003_endline', 'AM004_endline', 'AM005_endline']


In [57]:
print(
    "Cluster IDs consistent:",
    (
        baseline_endline["cc_code_baseline"]
        == baseline_endline["cc_code_endline"]
    ).all()
)

print(
    "Unique baseline clusters:",
    baseline_endline["cc_code_baseline"].nunique()
)

print(
    "Unique endline clusters:",
    baseline_endline["cc_code_endline"].nunique()
)

Cluster IDs consistent: True
Unique baseline clusters: 78
Unique endline clusters: 78


In [58]:
import statsmodels.formula.api as smf

outcomes = [
    ("Cog_Total", "TRS_C"),
    ("Rec_Total", "TRS_RL"),
    ("Exp_Total", "TRS_EL")
]

for endline_col, baseline_col in outcomes:
    formula = f"{endline_col} ~ treat + {baseline_col}"

    model = smf.ols(
        formula,
        data=baseline_endline
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": baseline_endline["cc_code_baseline"]
        }
    )

    ci = model.conf_int().loc["treat"]

    print(f"\n{'=' * 45}")
    print(f"Outcome: {endline_col}")
    print(f"Treatment Effect: {model.params['treat']:.4f}")
    print(f"Cluster-Robust SE: {model.bse['treat']:.4f}")
    print(f"P-value: {model.pvalues['treat']:.4f}")
    print(f"95% CI: [{ci.iloc[0]:.4f}, {ci.iloc[1]:.4f}]")


Outcome: Cog_Total
Treatment Effect: 0.3550
Cluster-Robust SE: 0.4822
P-value: 0.4616
95% CI: [-0.5900, 1.3001]

Outcome: Rec_Total
Treatment Effect: 0.8202
Cluster-Robust SE: 0.4870
P-value: 0.0922
95% CI: [-0.1343, 1.7746]

Outcome: Exp_Total
Treatment Effect: 0.5205
Cluster-Robust SE: 0.3806
P-value: 0.1714
95% CI: [-0.2254, 1.2665]


In [59]:
display(
    baseline[["TRS_C", "TRS_RL", "TRS_EL"]].dtypes
)

display(
    baseline[["TRS_C", "TRS_RL", "TRS_EL"]].head(10)
)

TRS_C     int64
TRS_RL    int64
TRS_EL    int64
dtype: object

,TRS_C,TRS_RL,TRS_EL
0,36,11,14
1,43,14,18
2,23,8,5
3,25,8,11
4,30,11,14
5,41,12,8
6,36,14,17
7,36,11,14
8,33,10,5
9,30,11,11


In [60]:
print("Baseline outcome columns:")
print(["TRS_C", "TRS_RL", "TRS_EL"])

print("\nEndline outcome columns:")
print(["Cog_Total", "Rec_Total", "Exp_Total"])

print("\nBaseline score ranges:")
display(baseline[["TRS_C", "TRS_RL", "TRS_EL"]].agg(["min", "max"]))

print("\nEndline score ranges:")
display(endline[["Cog_Total", "Rec_Total", "Exp_Total"]].agg(["min", "max"]))

Baseline outcome columns:
['TRS_C', 'TRS_RL', 'TRS_EL']

Endline outcome columns:
['Cog_Total', 'Rec_Total', 'Exp_Total']

Baseline score ranges:


,TRS_C,TRS_RL,TRS_EL
min,4,1,1
max,64,27,28



Endline score ranges:


,Cog_Total,Rec_Total,Exp_Total
min,0,0,0
max,81,47,47


In [61]:
print("Baseline:")
print(baseline[["TRS_C", "TRS_RL", "TRS_EL"]].agg(["min", "max"]))

print("\nEndline:")
print(endline[["Cog_Total", "Rec_Total", "Exp_Total"]].agg(["min", "max"]))

Baseline:


     TRS_C  TRS_RL  TRS_EL
min      4       1       1
max     64      27      28

Endline:
     Cog_Total  Rec_Total  Exp_Total
min          0          0          0
max         81         47         47


In [62]:
summary = baseline_endline.groupby("treatment").agg(
    Baseline_Cognitive=("TRS_C", "mean"),
    Endline_Cognitive=("Cog_Total", "mean"),
    Baseline_Receptive=("TRS_RL", "mean"),
    Endline_Receptive=("Rec_Total", "mean"),
    Baseline_Expressive=("TRS_EL", "mean"),
    Endline_Expressive=("Exp_Total", "mean"),
)

display(summary.round(2))

,Baseline_Cognitive,Endline_Cognitive,Baseline_Receptive,Endline_Receptive,Baseline_Expressive,Endline_Expressive
treatment,,,,,,
0,39.89,64.72,13.93,29.81,14.40,31.78
1,39.31,64.95,13.71,30.55,14.17,32.23


In [63]:
outcomes = [
    ("Cognitive", "TRS_C", "Cog_Total"),
    ("Receptive Language", "TRS_RL", "Rec_Total"),
    ("Expressive Language", "TRS_EL", "Exp_Total")
]

for name, baseline_col, endline_col in outcomes:
    means = baseline_endline.groupby("treatment")[
        [baseline_col, endline_col]
    ].mean()

    control_change = (
        means.loc[0, endline_col] - means.loc[0, baseline_col]
    )

    treatment_change = (
        means.loc[1, endline_col] - means.loc[1, baseline_col]
    )

    did = treatment_change - control_change

    print(f"\n{name}")
    print(f"Control change: {control_change:.3f}")
    print(f"Treatment change: {treatment_change:.3f}")
    print(f"Difference-in-Differences: {did:.3f}")


Cognitive
Control change: 24.823
Treatment change: 25.645
Difference-in-Differences: 0.822

Receptive Language
Control change: 15.888
Treatment change: 16.839
Difference-in-Differences: 0.951

Expressive Language
Control change: 17.379
Treatment change: 18.067
Difference-in-Differences: 0.688


In [64]:
import statsmodels.formula.api as smf

outcomes = [
    ("Cog_Total", "TRS_C"),
    ("Rec_Total", "TRS_RL"),
    ("Exp_Total", "TRS_EL")
]

for endline_col, baseline_col in outcomes:
    formula = f"{endline_col} ~ treat + {baseline_col} + C(union)"

    model = smf.ols(
        formula,
        data=baseline_endline
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": baseline_endline["cc_code_baseline"]
        }
    )

    ci = model.conf_int().loc["treat"]

    print(f"\n{'=' * 45}")
    print(f"Outcome: {endline_col}")
    print(f"Treatment Effect: {model.params['treat']:.4f}")
    print(f"Cluster-Robust SE: {model.bse['treat']:.4f}")
    print(f"P-value: {model.pvalues['treat']:.4f}")
    print(f"95% CI: [{ci.iloc[0]:.4f}, {ci.iloc[1]:.4f}]")


Outcome: Cog_Total
Treatment Effect: 0.5085
Cluster-Robust SE: 0.2920
P-value: 0.0816
95% CI: [-0.0638, 1.0809]

Outcome: Rec_Total
Treatment Effect: 0.8926
Cluster-Robust SE: 0.3093
P-value: 0.0039
95% CI: [0.2863, 1.4988]

Outcome: Exp_Total
Treatment Effect: 0.5499
Cluster-Robust SE: 0.2598
P-value: 0.0343
95% CI: [0.0408, 1.0590]


In [65]:
print("Unique union categories:", baseline_endline["union"].nunique())

print("\nChildren per union:")
display(
    baseline_endline.groupby("union")
    .size()
    .sort_values(ascending=False)
)

print("\nUnique treatment clusters:", baseline_endline["cc_code_baseline"].nunique())

Unique union categories: 32

Children per union:


union
20    159
5     130
32    127
7     127
30     98
26     98
11     97
16     96
18     96
19     95
17     92
22     86
14     66
13     66
4      66
28     66
2      65
29     65
31     65
27     65
3      65
10     65
8      64
1      64
15     64
21     63
9      63
25     59
6      57
23     33
12     32
24     32
dtype: int64


Unique treatment clusters: 78


In [66]:
print("Unique union categories:", baseline_endline["union"].nunique())

print("\nTreatment count by union:")
display(
    pd.crosstab(
        baseline_endline["union"],
        baseline_endline["treat"]
    )
)

Unique union categories: 32

Treatment count by union:


treat,0,1
union,,
1,32,32
2,32,33
3,32,33
4,33,33
5,65,65
6,29,28
7,62,65
8,31,33
9,31,32


In [67]:
print("Baseline union column exists:", "union_baseline" in baseline_endline.columns)
print("Endline union column exists:", "union" in baseline_endline.columns)

print("\nUnion categories:")
print(sorted(baseline_endline["union"].dropna().unique()))

print("\nMissing union values:")
print(baseline_endline["union"].isna().sum())

Baseline union column exists: False
Endline union column exists: True

Union categories:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32)]

Missing union values:
0


In [68]:
import pandas as pd
import statsmodels.formula.api as smf

outcomes = {
    "Cognitive": ("Cog_Total", "TRS_C"),
    "Receptive Language": ("Rec_Total", "TRS_RL"),
    "Expressive Language": ("Exp_Total", "TRS_EL")
}

results = []

for name, (endline_col, baseline_col) in outcomes.items():
    formula = f"{endline_col} ~ treat + {baseline_col} + C(union)"

    model = smf.ols(
        formula,
        data=baseline_endline
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": baseline_endline["cc_code_baseline"]}
    )

    results.append({
        "Outcome": name,
        "Treatment Effect": model.params["treat"],
        "Cluster SE": model.bse["treat"],
        "P-value": model.pvalues["treat"],
        "CI Lower": model.conf_int().loc["treat", 0],
        "CI Upper": model.conf_int().loc["treat", 1]
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Outcome,Treatment Effect,Cluster SE,P-value,CI Lower,CI Upper
0,Cognitive,0.5085,0.2920,0.0816,-0.0638,1.0809
1,Receptive Language,0.8926,0.3093,0.0039,0.2863,1.4988
2,Expressive Language,0.5499,0.2598,0.0343,0.0408,1.0590


In [69]:
print("Merged dataset columns:\n")

for col in baseline_endline.columns:
    print(col)

Merged dataset columns:

V001
treatment
cc_code_baseline
upazila_baseline
V001a
V5
V16
V17
V18
V21y
V21m
V22m
V23
RC_a
RC_b
EC_a
EC_b
Sum_a
Sum_b
TRS_C
TRS_RL
TRS_EL
AM001_baseline
AM002_baseline
AM003_baseline
AM004_baseline
AM005_baseline
cc_code_endline
upazila_endline
district
union
treat
tc
TC_other
V004
V005
V016
V017
V018
V021_Y
V021_M
V022_M
V023
RC_1
RC_2
EC_1
EC_2
Sum_1
Sum_2
V024
V025
V026
V027
V028
Cog_Total
Rec_Total
Exp_Total
AM001_endline
AM002_endline
AM003_endline
AM004_endline
AM005_endline


In [70]:
keywords = [
    "gender", "sex", "age", "weight", "mother",
    "religion", "education", "employ", "father",
    "household", "asset", "shock", "death",
    "disaster", "clinic", "union"
]

for col in baseline_endline.columns:
    if any(word in col.lower() for word in keywords):
        print(col)

union


In [71]:
print(baseline_endline.columns.tolist())

['V001', 'treatment', 'cc_code_baseline', 'upazila_baseline', 'V001a', 'V5', 'V16', 'V17', 'V18', 'V21y', 'V21m', 'V22m', 'V23', 'RC_a', 'RC_b', 'EC_a', 'EC_b', 'Sum_a', 'Sum_b', 'TRS_C', 'TRS_RL', 'TRS_EL', 'AM001_baseline', 'AM002_baseline', 'AM003_baseline', 'AM004_baseline', 'AM005_baseline', 'cc_code_endline', 'upazila_endline', 'district', 'union', 'treat', 'tc', 'TC_other', 'V004', 'V005', 'V016', 'V017', 'V018', 'V021_Y', 'V021_M', 'V022_M', 'V023', 'RC_1', 'RC_2', 'EC_1', 'EC_2', 'Sum_1', 'Sum_2', 'V024', 'V025', 'V026', 'V027', 'V028', 'Cog_Total', 'Rec_Total', 'Exp_Total', 'AM001_endline', 'AM002_endline', 'AM003_endline', 'AM004_endline', 'AM005_endline']


In [72]:
# Inspect baseline variable labels and sample values

for col in baseline.columns:
    print(f"\nColumn: {col}")
    print("Data type:", baseline[col].dtype)
    print("Sample values:", baseline[col].dropna().head(5).tolist())


Column: V001
Data type: int64
Sample values: [540, 538, 552, 546, 537]

Column: treatment
Data type: int64
Sample values: [1, 1, 1, 1, 1]

Column: cc_code
Data type: int64
Sample values: [1, 1, 1, 1, 1]

Column: upazila
Data type: int64
Sample values: [1, 1, 1, 1, 1]

Column: V001a
Data type: int64
Sample values: [4, 4, 2, 4, 2]

Column: V5
Data type: int64
Sample values: [2, 1, 1, 2, 1]

Column: V16
Data type: int64
Sample values: [36, 43, 23, 25, 30]

Column: V17
Data type: int64
Sample values: [6, 6, 8, 7, 9]

Column: V18
Data type: int64
Sample values: [80, 80, 90, 85, 95]

Column: V21y
Data type: float64
Sample values: [1.0, 1.0, 1.0, 1.0, 1.0]

Column: V21m
Data type: float64
Sample values: [5.0, 5.0, 7.0, 7.0, 11.0]

Column: V22m
Data type: int64
Sample values: [12, 17, 5, 7, 7]

Column: V23
Data type: int64
Sample values: [9, 11, 5, 7, 7]

Column: RC_a
Data type: int64
Sample values: [11, 14, 8, 8, 11]

Column: RC_b
Data type: int64
Sample values: [5, 6, 8, 6, 10]

Column: EC_

In [73]:
# Show baseline variable names and unique-value counts
for col in baseline.columns:
    print(
        f"{col:15} | "
        f"Unique values: {baseline[col].nunique():5} | "
        f"Missing: {baseline[col].isna().sum()}"
    )

V001            | Unique values:  2574 | Missing: 0
treatment       | Unique values:     2 | Missing: 0
cc_code         | Unique values:    78 | Missing: 0
upazila         | Unique values:     3 | Missing: 0
V001a           | Unique values:     6 | Missing: 0
V5              | Unique values:     2 | Missing: 0
V16             | Unique values:    57 | Missing: 0
V17             | Unique values:    19 | Missing: 0
V18             | Unique values:    19 | Missing: 0
V21y            | Unique values:     1 | Missing: 1198
V21m            | Unique values:    11 | Missing: 222
V22m            | Unique values:    16 | Missing: 0
V23             | Unique values:    11 | Missing: 0
RC_a            | Unique values:    25 | Missing: 0
RC_b            | Unique values:    19 | Missing: 0
EC_a            | Unique values:    28 | Missing: 0
EC_b            | Unique values:    19 | Missing: 0
Sum_a           | Unique values:    34 | Missing: 0
Sum_b           | Unique values:    34 | Missing: 0
TRS_C  

In [74]:
for name, (endline_col, baseline_col) in {
    "Cognitive": ("V018", "V18"),
    "Language": ("Sum_2", "Sum_b")
}.items():

    cols = [endline_col, baseline_col, "treat", "cc_code_baseline"]
    df = baseline_endline[cols].dropna()

    model = smf.ols(
        f"{endline_col} ~ treat + {baseline_col}",
        data=df
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["cc_code_baseline"]}
    )

    print(f"\n{name} | N={len(df)}")
    print("Effect:", round(model.params["treat"], 4))
    print("SE:", round(model.bse["treat"], 4))
    print("P-value:", round(model.pvalues["treat"], 4))
    print("95% CI:", model.conf_int().loc["treat"].round(4).tolist())


Cognitive | N=2486
Effect: 0.7668
SE: 0.7269
P-value: 0.2915
95% CI: [-0.6579, 2.1915]

Language | N=2482
Effect: 2.147
SE: 0.9491
P-value: 0.0237
95% CI: [0.2868, 4.0072]


## Final Results and Conclusion

### Methodology
We used Analysis of Covariance (ANCOVA) to estimate the treatment effect on child development outcomes. The models controlled for baseline scores, and standard errors were clustered at the community-clinic level.

### Results
- **Cognitive Outcome:** Estimated effect = +0.7668, p-value = 0.2915. The effect was not statistically significant.
- **Language Outcome:** Estimated effect = +2.1470, p-value = 0.0237. The effect was positive and statistically significant at the 5% level.

### Limitations
1. The comparability of baseline and endline score scales requires further verification.
2. The union categories in the available dataset do not yet match the categories described in the published report.
3. The estimates should be treated as preliminary until the outcome definitions and model specification are fully verified.

### Conclusion
The preliminary analysis suggests a positive effect on language outcomes, while the estimated cognitive effect was not statistically significant. These findings should not be interpreted as definitive causal estimates until the data definitions and model specification have been verified.